<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Scatter Plot**


Estimated time needed: **45** minutes


## Overview

In this lab, you will focus on creating and interpreting scatter plots to visualize relationships between variables and trends in the dataset. The provided dataset will be directly loaded into a pandas DataFrame, and various scatter plot-related visualizations will be created to explore developer trends, compensation, and preferences.



## Objectives


In this lab, you will:

- Create and analyze scatter plots to examine relationships between variables.

- Use scatter plots to identify trends and patterns in the dataset.

- Focus on visualizations centered on scatter plots for better data-driven insights.


## Setup: Working with the Database



**Install and import the required libraries**


In [ ]:
!pip install pandas
!pip install matplotlib

import pandas as pd
import matplotlib.pyplot as plt

#### Step 1: Load the dataset


In [ ]:
file_path = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"

df = pd.read_csv(file_path)



### Task 1: Exploring Relationships with Scatter Plots



#### 1. Scatter Plot for Age vs. Job Satisfaction



Visualize the relationship between respondents' age (`Age`) and job satisfaction (`JobSatPoints_6`). Use this plot to identify any patterns or trends.




In [ ]:
# ---- Helpers used throughout this lab ----
import numpy as np

# Age is stored as text ranges, so map each range to an approximate number (its midpoint)
AGE_MAP = {
    'Under 18 years old': 17, '18-24 years old': 21, '25-34 years old': 29.5,
    '35-44 years old': 39.5, '45-54 years old': 49.5, '55-64 years old': 59.5,
    '65 years or older': 70,
}
AGE_ORDER = list(AGE_MAP.keys())

# TimeSearching / TimeAnswering are text ranges -> approximate minutes per day
TIME_MAP = {
    'Less than 15 minutes a day': 7.5, '15-30 minutes a day': 22.5,
    '30-60 minutes a day': 45, '60-120 minutes a day': 90, 'Over 120 minutes a day': 150,
}

def years_to_num(series):
    """YearsCodePro is text: turn 'Less than 1 year' / 'More than 50 years' into numbers."""
    return pd.to_numeric(series.replace({'Less than 1 year': '0', 'More than 50 years': '51'}),
                         errors='coerce')

def split_counts(series, top_n=None):
    """Count items in a ';'-separated multi-answer column."""
    counts = series.dropna().str.split(';').explode().str.strip().value_counts()
    return counts.head(top_n) if top_n else counts

df['Age_numeric'] = df['Age'].map(AGE_MAP)
df['JobSatPoints_6'] = pd.to_numeric(df['JobSatPoints_6'], errors='coerce')
df['YearsCodePro_num'] = years_to_num(df['YearsCodePro'])
comp_limit = df['ConvertedCompYearly'].quantile(0.99)   # used to trim extreme salaries from the charts

# ---- 1.1 Age vs JobSatPoints_6 ----
d = df[['Age_numeric', 'JobSatPoints_6']].dropna()
jitter = np.random.uniform(-2, 2, len(d))   # Age only has a few values, jitter spreads the points

plt.figure(figsize=(10, 6))
plt.scatter(d['Age_numeric'] + jitter, d['JobSatPoints_6'], alpha=0.2)
plt.title('Age vs Job Satisfaction (JobSatPoints_6)')
plt.xlabel('Age (approximate)')
plt.ylabel('JobSatPoints_6')
plt.show()

#### 2. Scatter Plot for Compensation vs. Job Satisfaction


Explore the relationship between yearly compensation (`ConvertedCompYearly`) and job satisfaction (`JobSatPoints_6`) using a scatter plot.


In [ ]:
# ---- 1.2 ConvertedCompYearly vs JobSatPoints_6 ----
d = df[['ConvertedCompYearly', 'JobSatPoints_6']].dropna()
d = d[d['ConvertedCompYearly'] <= comp_limit]

plt.figure(figsize=(10, 6))
plt.scatter(d['ConvertedCompYearly'], d['JobSatPoints_6'], alpha=0.2, color='seagreen')
plt.title('Yearly Compensation vs Job Satisfaction')
plt.xlabel('ConvertedCompYearly (USD)')
plt.ylabel('JobSatPoints_6')
plt.show()
print("Correlation:", round(d['ConvertedCompYearly'].corr(d['JobSatPoints_6']), 3))

### Task 2: Enhancing Scatter Plots


#### 1. Scatter Plot with Trend Line for Age vs. Job Satisfaction



Add a regression line to the scatter plot of Age vs. JobSatPoints_6 to highlight trends in the data.


In [ ]:
# ---- 2.1 Age vs JobSatPoints_6 with a trend (regression) line ----
d = df[['Age_numeric', 'JobSatPoints_6']].dropna()
slope, intercept = np.polyfit(d['Age_numeric'], d['JobSatPoints_6'], 1)
x_line = np.linspace(d['Age_numeric'].min(), d['Age_numeric'].max(), 100)

plt.figure(figsize=(10, 6))
plt.scatter(d['Age_numeric'] + np.random.uniform(-2, 2, len(d)), d['JobSatPoints_6'], alpha=0.2)
plt.plot(x_line, slope * x_line + intercept, color='red', linewidth=2,
         label=f'Trend: y = {slope:.2f}x + {intercept:.1f}')
plt.title('Age vs Job Satisfaction with Trend Line')
plt.xlabel('Age (approximate)')
plt.ylabel('JobSatPoints_6')
plt.legend()
plt.show()

#### 2. Scatter Plot for Age vs. Work Experience


Visualize the relationship between Age (`Age`) and Work Experience (`YearsCodePro`) using a scatter plot.


In [ ]:
# ---- 2.2 Age vs YearsCodePro ----
d = df[['Age_numeric', 'YearsCodePro_num']].dropna()

plt.figure(figsize=(10, 6))
plt.scatter(d['Age_numeric'] + np.random.uniform(-2, 2, len(d)), d['YearsCodePro_num'], alpha=0.2, color='purple')
plt.title('Age vs Years of Professional Coding Experience')
plt.xlabel('Age (approximate)')
plt.ylabel('YearsCodePro')
plt.show()

### Task 3: Combining Scatter Plots with Additional Features


#### 1. Bubble Plot of Compensation vs. Job Satisfaction with Age as Bubble Size



Create a bubble plot to explore the relationship between yearly compensation (`ConvertedCompYearly`) and job satisfaction (`JobSatPoints_6`), with bubble size representing age.


In [ ]:
# ---- 3.1 Bubble plot: compensation vs job satisfaction, bubble size = age ----
d = df[['ConvertedCompYearly', 'JobSatPoints_6', 'Age_numeric']].dropna()
d = d[d['ConvertedCompYearly'] <= comp_limit]

plt.figure(figsize=(12, 7))
plt.scatter(d['ConvertedCompYearly'], d['JobSatPoints_6'], s=d['Age_numeric'] * 2,
            alpha=0.3, color='steelblue', edgecolors='w', linewidth=0.5)
plt.title('Compensation vs Job Satisfaction (bubble size = age)')
plt.xlabel('ConvertedCompYearly (USD)')
plt.ylabel('JobSatPoints_6')
plt.show()

#### 2. Scatter Plot for Popular Programming Languages by Job Satisfaction


Visualize the popularity of programming languages (`LanguageHaveWorkedWith`) against job satisfaction using a scatter plot. Use points to represent satisfaction levels for each language.


In [ ]:
# ---- 3.2 Popular languages vs job satisfaction ----
lang = df[['LanguageHaveWorkedWith', 'JobSatPoints_6']].dropna()
lang = lang.assign(Language=lang['LanguageHaveWorkedWith'].str.split(';')).explode('Language')
summary = lang.groupby('Language').agg(Users=('JobSatPoints_6', 'size'),
                                       Median_Sat=('JobSatPoints_6', 'median'))
top = summary.sort_values('Users', ascending=False).head(15)

plt.figure(figsize=(12, 7))
plt.scatter(top['Users'], top['Median_Sat'], s=120, color='darkorange')
for name, row in top.iterrows():
    plt.annotate(name, (row['Users'], row['Median_Sat']), xytext=(5, 3), textcoords='offset points')
plt.title('Top 15 Languages: Number of Users vs Median Job Satisfaction')
plt.xlabel('Number of respondents who use the language')
plt.ylabel('Median JobSatPoints_6')
plt.show()

### Task 4: Scatter Plot Comparisons Across Groups


#### 1. Scatter Plot for Compensation vs. Job Satisfaction by Employment Type


Visualize the relationship between yearly compensation (`ConvertedCompYearly`) and job satisfaction (`JobSatPoints_6`), categorized by employment type (`Employment`). Use color coding or markers to differentiate between employment types.


In [ ]:
# ---- 4.1 Compensation vs job satisfaction by employment type ----
d = df[['ConvertedCompYearly', 'JobSatPoints_6', 'Employment']].dropna()
d = d[d['ConvertedCompYearly'] <= comp_limit]
top_emp = d['Employment'].value_counts().head(5).index

plt.figure(figsize=(12, 7))
for emp in top_emp:
    sub = d[d['Employment'] == emp]
    plt.scatter(sub['ConvertedCompYearly'], sub['JobSatPoints_6'], alpha=0.4, label=emp)
plt.title('Compensation vs Job Satisfaction by Employment Type')
plt.xlabel('ConvertedCompYearly (USD)')
plt.ylabel('JobSatPoints_6')
plt.legend(title='Employment', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

#### 2. Scatter Plot for Work Experience vs. Age Group by Country


Compare work experience (`YearsCodePro`) across different age groups (`Age`) and countries (`Country`). Use colors to represent different countries and markers for age groups.


In [ ]:
# ---- 4.2 Work experience vs age group, coloured by country ----
d = df[['Age', 'Age_numeric', 'YearsCodePro_num', 'Country']].dropna()
top_countries = d['Country'].value_counts().head(5).index
d = d[d['Country'].isin(top_countries)]

markers = dict(zip(AGE_ORDER, ['v', 'o', 's', '^', 'D', 'P', '*']))
colors = dict(zip(top_countries, plt.cm.tab10.colors))

plt.figure(figsize=(12, 7))
for (country, age), sub in d.groupby(['Country', 'Age']):
    plt.scatter(sub['Age_numeric'] + np.random.uniform(-2, 2, len(sub)), sub['YearsCodePro_num'],
                color=colors[country], marker=markers.get(age, 'o'), alpha=0.4)
# Legend: one entry per country (colour) and per age group (marker)
for country, col in colors.items():
    plt.scatter([], [], color=col, label=country)
for age, m in markers.items():
    plt.scatter([], [], color='gray', marker=m, label=age)
plt.title('Work Experience vs Age Group by Country (top 5 countries)')
plt.xlabel('Age (approximate)')
plt.ylabel('YearsCodePro')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Final Step: Review


With these scatter plots, you will have analyzed data relationships across multiple dimensions, including compensation, job satisfaction, employment types, and demographics, to uncover meaningful trends in the developer community.


### Summary


After completing this lab, you will be able to:
- Analyze how numerical variables relate across specific groups, such as employment types and countries.
- Use scatter plots effectively to represent multiple variables with color, size, and markers.
- Gain insights into compensation, satisfaction, and demographic trends using advanced scatter plot techniques.


## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


<!--## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|               
|2024-10-07|1.2|Madhusudan Moole|Reviewed and updated lab|                                                                                      
|2024-10-06|1.0|Raghul Ramesh|Created lab|-->


Copyright © IBM Corporation. All rights reserved.
